In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
from torch.nn import *
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from constants import *
from model_gnn import PositionalEncoding, SequenceEncoder
import gc

torch.set_default_dtype(torch.float32)
# DEVICE1/DEVICE2 imported from constants

In [2]:
mhc_df = pd.read_parquet('data/mhc_db.parquet')
mhc_df = mhc_df.loc[(mhc_df['sequence'].str.len() <= 450)]
mhc_df

,allele,sequence
0,Aole-DPB1*28:01,MVLQVSGPPQTVALTALLMVLLTPVVQGRATPENYVYQRLYECYAF...
1,Aole-DPB1*28:02,MVLQVSGPPQTVALTALLMVLLTPVVQGRATPENYVYQRLYECYAF...
2,Aole-DPB1*29:01,MVLQVSGPPHTVALTALLMVLLTPVVQGRATPENYVYQRLYECYAF...
3,Aole-DQA1*27:01,MILNKALMLGALILTTVMSPCGGEDIVADHVASYGVNLYQSYGPSG...
4,Aole-DQA1*27:02,MILNKALMLGALILTTVMSPCGGEDIVADHVASYGINLYQSYGPSG...
...,...,...
10389,Tutr-DQB*001:01,MSGTVALQIPRGLWTTAVMVMLTVLSTPEAEGRDSPQDFLYRYMFM...
10390,Tutr-DRA*001:01,MAIIGVPIPGFFIIVLISLQESWAITEDHVIIQAEFSLSPYQSNEF...
10391,Tutr-DRB1*001:01,MVSLYFSGGSWMAALTVILMVLSPPLAWTRETPSLFMYQFKSECHF...
10392,Tutr-N*001:01,MLWVMAARTLLLLLTGAMTLTETWAGSHSLRYFYTGVSRPGRGEPR...


In [13]:
MAX_LEN_MHC = mhc_df['sequence'].str.len().max() + 2
MAX_LEN_MHC

np.int64(403)

In [4]:
class MHCDataset(Dataset):
    def __init__(self, seq_features, seq_targets, token_targets):
        self.seq_features = torch.tensor(seq_features, dtype=torch.int32)
        self.seq_targets = torch.tensor(seq_targets, dtype=torch.int32)
        self.token_targets = torch.tensor(token_targets, dtype=torch.float32)

    def __len__(self):
        return len(self.seq_features)

    def __getitem__(self, idx):
        return self.seq_features[idx], self.seq_targets[idx], self.token_targets[idx]

In [2]:
gc.collect()
torch.cuda.empty_cache()       


class MHCAutoencoder(Module):
    def __init__(
        self,
        vocab_size=45,
        embedding_dim=128,
        dim_feedforward=1024,
        n_heads=8,
        n_layers=6,
        dropout=0.1,
    ):
        super(MHCAutoencoder, self).__init__()
        self.dropout = Dropout(dropout)
        self.embedding = Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.positional_encoding = PositionalEncoding(embedding_dim)
        self.encoder = SequenceEncoder(embedding_dim, dim_feedforward, n_heads, n_layers, dropout, DEVICE1)

        decoder_layer = TransformerDecoderLayer(
            d_model=embedding_dim,
            nhead=4,
            batch_first=True,
            dim_feedforward=512,
            dropout=dropout
        ).to(DEVICE1)
        self.decoder = TransformerDecoder(
            decoder_layer=decoder_layer,
            num_layers=1,
            norm=LayerNorm(embedding_dim),
        )
        
        self.bn_decoder = BatchNorm1d(embedding_dim)
        self.fc_out = Linear(embedding_dim, len(AMINO_ACIDS))


    def forward(self, src, tgt):
        src_padding_mask = src == 0
        tgt_padding_mask = tgt == 0
        
        src = self.embedding(src)
        src = self.positional_encoding(src)
        
        memory = self.encoder(src, src_padding_mask)

        tgt = self.embedding(tgt)
        tgt = self.positional_encoding(tgt)
        
        x = self.decoder(tgt, memory.unsqueeze(1), tgt_key_padding_mask=tgt_padding_mask)[:, 0, :]
        x = F.leaky_relu(self.bn_decoder(x))
        x = torch.log_softmax(self.fc_out(x), -1)
        return x, memory


def model_predict(m, src, tgt, token):
    out, encoding = m(src, tgt)
    loss = F.cross_entropy(out, token)
    acc = accuracy_score(np.argmax(token.cpu().detach().numpy(), -1), np.argmax(out.cpu().detach().numpy(), -1))
    return loss, acc, encoding

In [6]:
sequences = []
for _, row in tqdm(mhc_df.iterrows(), total=len(mhc_df)):
    seq = row['sequence']
    for i in range(len(seq)):
        src = list(seq)
        src = ['[CLS]'] + src + ['[SEP]']
        src = [TOKEN_VOCABULARY[e] for e in src] + [0 for _ in range(MAX_LEN_MHC - len(src))]
        tgt = ['[CLS]'] + list(seq[:len(seq)-i-1]) + ['[SEP]']
        tgt = [TOKEN_VOCABULARY[e] for e in tgt] + [0 for _ in range(MAX_LEN_MHC - len(tgt))]
        token = seq[len(seq)-i-1]
        sequences.append([seq, src, tgt, token])

string_sequences, input_sequences, output_sequences, token_cls = list(zip(*sequences))
del sequences
token_cls = (np.expand_dims(token_cls, -1) == AMINO_ACIDS) * 1

string_sequences = pd.Series(np.array(string_sequences))
input_sequences = np.array(input_sequences)
output_sequences = np.array(output_sequences)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 33638/33638 [09:25<00:00, 59.50it/s]


In [7]:
mask = ~pd.DataFrame(np.concatenate([input_sequences, output_sequences], axis=-1)).duplicated(keep=False)
string_sequences = string_sequences[mask]
input_sequences = input_sequences[mask]
output_sequences = output_sequences[mask]
token_cls = token_cls[mask]

## Training

In [ ]:
NAME = 'pretrain_mhc'
EPOCHS = 30
BATCH_SIZE = 100

train_data = MHCDataset(input_sequences, output_sequences, token_cls)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

model = MHCAutoencoder(dropout=0.05, embedding_dim=128, dim_feedforward=1024, n_heads=8, n_layers=6).to(DEVICE1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(EPOCHS+1):
    model.train()
    batches = tqdm(train_loader)
    l, accs = [], []
    for src_seq, tgt_seq, tgt_token in batches:
        src_seq, tgt_seq, tgt_token = src_seq.to(DEVICE1), tgt_seq.to(DEVICE1), tgt_token.to(DEVICE1)
        optimizer.zero_grad()
        loss, acc, _ = model_predict(model, src_seq, tgt_seq, tgt_token)
        l.append(loss.item())
        loss.backward()
        optimizer.step()
        accs.append(acc)
        batches.set_description(f'Train epoch {epoch} - loss: {np.mean(l[-100:]):.4f}, acc: {np.mean(accs[-100:]):.4f}')

torch.save(model.state_dict(), f'weights/{NAME}.pt')